# Decision Tree Impurity Metrics and Recursion

Four topics covered:

1. **Impurity metrics from scratch** — misclassification rate, Gini index, entropy for binary splits
2. **Generalised to k classes** — same framework extended to 4-class problems
3. **Gini index proof** — showing $L_{Gini} = 1 - \sum_i \hat{p}_i^2$ is equivalent to the expanded form
4. **Recursion** — Fibonacci sequence and palindrome checker as recursive functions

No external libraries needed beyond `math` — all metrics implemented from first principles.

In [ ]:
import math

---
## 1. Impurity Metrics for Binary Splits

For a node with class counts $(n_1, n_2)$, the three impurity measures are:

$$\text{Misclassification} = 1 - \max(p_1, p_2)$$

$$\text{Gini} = 1 - (p_1^2 + p_2^2)$$

$$\text{Entropy} = -\sum_i p_i \log_2 p_i \quad (\text{terms with } p_i = 0 \text{ omitted})$$

**Impurity reduction** for a split: parent impurity minus the weighted average of child impurities, where weights are the fraction of samples going to each child.

In [ ]:
def misclassification(p1, p2):
    """1 - max class probability. Minimised when one class dominates."""
    return 1 - max(p1, p2)

def gini(p1, p2):
    """1 - sum of squared probabilities. Maximised at uniform distribution."""
    return 1 - (p1**2 + p2**2)

def entropy(p1, p2):
    """Shannon entropy in bits. Terms with p=0 contribute 0 (limit convention)."""
    e = 0
    if p1 > 0: e -= p1 * math.log2(p1)
    if p2 > 0: e -= p2 * math.log2(p2)
    return e

def impurity_change(parent, left, right, method):
    """
    Compute impurity reduction for a binary split.
    
    parent, left, right: tuples of (class_0_count, class_1_count)
    method: 'mis' | 'gini' | 'entropy'
    Returns: parent_impurity - weighted_child_impurity
    """
    p_total = sum(parent)
    l_total = sum(left)
    r_total = sum(right)

    p1, p2   = parent[0]/p_total, parent[1]/p_total
    lp1, lp2 = left[0]/l_total,   left[1]/l_total
    rp1, rp2 = right[0]/r_total,  right[1]/r_total

    fn = {'mis': misclassification, 'gini': gini, 'entropy': entropy}[method]

    parent_imp  = fn(p1, p2)
    weighted    = (l_total/p_total) * fn(lp1, lp2) + (r_total/p_total) * fn(rp1, rp2)

    return parent_imp - weighted

In [ ]:
parent = (60, 40)
splits = [
    ((35, 5),  (25, 35)),   # Split 1 — large asymmetry in both children
    ((40, 10), (20, 30)),   # Split 2 — moderate asymmetry
    ((45, 25), (15, 15)),   # Split 3 — right child perfectly balanced
]

print(f"{'Split':<8} {'Misclassification':>20} {'Gini':>12} {'Entropy':>12}")
print("-" * 56)
for i, (L, R) in enumerate(splits, 1):
    mis = impurity_change(parent, L, R, 'mis')
    gin = impurity_change(parent, L, R, 'gini')
    ent = impurity_change(parent, L, R, 'entropy')
    print(f"Split {i}  {mis:>20.6f} {gin:>12.6f} {ent:>12.6f}")

print("""
Observation:
Split 1 achieves the highest impurity reduction across all three measures.
Split 3 produces near-zero reduction — the right child (15,15) is perfectly impure,
offsetting most of the gain from the left child (45,25).
Entropy is the most sensitive measure, showing the largest spread between splits.
""")

---
## 2. Impurity Metrics Generalised to k Classes

The same framework extends naturally to any number of classes. For a node with class proportions $p_1, \ldots, p_C$:

$$\text{Misclassification} = 1 - \max_i(p_i)$$

$$\text{Gini} = 1 - \sum_{i=1}^C p_i^2$$

$$\text{Entropy} = -\sum_{i=1}^C p_i \log_2 p_i$$

The functions below accept a list of probabilities of any length, making them class-count agnostic.

In [ ]:
def misclassification_k(probs):
    """Misclassification rate for k classes."""
    return 1 - max(probs)

def gini_k(probs):
    """Gini impurity for k classes: 1 - sum(p_i^2)."""
    return 1 - sum(p*p for p in probs)

def entropy_k(probs):
    """Shannon entropy for k classes. Skips zero-probability terms."""
    return -sum(p * math.log2(p) for p in probs if p > 0)

def impurity_change_k(parent_counts, left_counts, right_counts, method):
    """
    Generalised impurity reduction for k-class splits.
    Counts are lists of per-class sample counts.
    """
    total = sum(parent_counts)
    lt    = sum(left_counts)
    rt    = sum(right_counts)

    p_probs = [c/total for c in parent_counts]
    l_probs = [c/lt    for c in left_counts]
    r_probs = [c/rt    for c in right_counts]

    fn = {'mis': misclassification_k, 'gini': gini_k, 'entropy': entropy_k}[method]

    parent_imp = fn(p_probs)
    weighted   = (lt/total)*fn(l_probs) + (rt/total)*fn(r_probs)
    return parent_imp - weighted

In [ ]:
# 4-class split: parent (25,25,25,25) → left (0,20,10,10) | right (25,5,15,15)
parent = [25, 25, 25, 25]
left   = [0,  20, 10, 10]
right  = [25,  5, 15, 15]

print("4-class split: (25,25,25,25) → (0,20,10,10) | (25,5,15,15)")
print(f"  Misclassification reduction : {impurity_change_k(parent, left, right, 'mis'):.6f}")
print(f"  Gini reduction              : {impurity_change_k(parent, left, right, 'gini'):.6f}")
print(f"  Entropy reduction           : {impurity_change_k(parent, left, right, 'entropy'):.6f}")
print()
print("""
Note: The parent (25,25,25,25) is maximally impure — all classes equally likely.
The left child (0,20,10,10) has eliminated class 0 entirely, reducing impurity.
The right child (25,5,15,15) is less pure but not balanced.
Entropy shows the largest absolute reduction because it is most sensitive to
the elimination of class 0 from the left child.
""")

---
## 3. Gini Index: Compact Form Proof

**Question:** Can the Gini index be written as $L_{Gini} = 1 - \sum_{i=1}^{C} \hat{p}_i^2$?

**Yes — this is exactly the standard definition.** The proof is direct:

The Gini impurity measures the probability that two samples drawn at random from the node have *different* class labels. For class $i$ with proportion $\hat{p}_i$:

$$L_{Gini} = \sum_{i=1}^C \hat{p}_i (1 - \hat{p}_i) = \sum_{i=1}^C \hat{p}_i - \sum_{i=1}^C \hat{p}_i^2 = 1 - \sum_{i=1}^C \hat{p}_i^2$$

where the last step uses $\sum_i \hat{p}_i = 1$. The compact summation form $1 - \sum_i \hat{p}_i^2$ is therefore algebraically identical to the expanded $1 - (p_1^2 + p_2^2 + \ldots + p_C^2)$ — it simply uses $\Sigma$ notation to handle any number of classes without listing each term.

In [ ]:
def gini_general(probs):
    """Gini = 1 - sum(p_i^2), works for any number of classes."""
    return 1 - sum(p**2 for p in probs)

# Verify on known cases
cases = {
    "Binary (0.6, 0.4)"       : [0.6, 0.4],
    "Uniform 4-class"         : [0.25, 0.25, 0.25, 0.25],
    "Pure node (1.0)"         : [1.0, 0.0],
    "Max impure binary"       : [0.5, 0.5],
}

for label, probs in cases.items():
    g = gini_general(probs)
    print(f"{label:<30} Gini = {g:.4f}")

print()
print("Pure node → Gini = 0 (minimum impurity)")
print("Uniform k-class → Gini = 1 - 1/k (maximum impurity for k classes)")

---
## 4. Recursion

Every recursive function has three parts:
- **Input** — the argument(s) passed to the function
- **Terminating condition** — the base case(s) where recursion stops
- **Recursive step** — the call to itself with a strictly smaller/simpler input

### 4.1 Fibonacci sequence

| Part | Description |
|---|---|
| Input | Integer $n \geq 0$ |
| Terminating conditions | $F(0) = 0$, $F(1) = 1$ |
| Recursive step | $F(n) = F(n-1) + F(n-2)$ — reduces to two strictly smaller subproblems |

In [ ]:
def fib(n: int) -> int:
    """
    Return the nth Fibonacci number recursively.
    
    Input            : n >= 0
    Terminating cond : n == 0 → 0, n == 1 → 1
    Recursive step   : fib(n) = fib(n-1) + fib(n-2)
    """
    if n == 0: return 0          # base case 1
    if n == 1: return 1          # base case 2
    return fib(n-1) + fib(n-2)  # recursive step — n shrinks toward base cases

def fibonacci_list(n: int) -> list:
    """Generate the first n Fibonacci numbers."""
    return [fib(i) for i in range(n)]

print("First 10 Fibonacci numbers:", fibonacci_list(10))
print()
print("Trace for fib(5):")
print("  fib(5) = fib(4) + fib(3)")
print("         = (fib(3) + fib(2)) + (fib(2) + fib(1))")
print("         = ((fib(2)+fib(1)) + (fib(1)+fib(0))) + ((fib(1)+fib(0)) + 1)")
print("         = ((1+1) + (1+0)) + ((1+0) + 1) = 3 + 2 = 5  ✓")

### 4.2 Palindrome checker

A string is a palindrome if it reads the same forwards and backwards. The recursive insight: after checking that the first and last characters match, the remaining problem is identical but on a shorter string.

| Part | Description |
|---|---|
| Input | String `s` |
| Terminating conditions | `len(s) <= 1` — empty string or single character is always a palindrome |
| Recursive step | If `s[0] == s[-1]`: check `is_palindrome(s[1:-1])` — strips outer characters, reducing length by 2 |

In [ ]:
def is_palindrome(s: str) -> bool:
    """
    Check if string s is a palindrome recursively.
    
    Input            : string s
    Terminating cond : len(s) <= 1  (empty or single char → always palindrome)
    Recursive step   : s[0] == s[-1] and is_palindrome(s[1:-1])
                       strips outermost characters, problem shrinks by 2 each call
    """
    if len(s) <= 1:    return True   # base case
    if s[0] != s[-1]:  return False  # early exit — mismatch found
    return is_palindrome(s[1:-1])    # recursive step on inner substring

# Test cases
test_words = ["madam", "racecar", "hello", "level", "python", "a", ""]
for word in test_words:
    result = is_palindrome(word)
    print(f"  is_palindrome({word!r:<10}) = {result}")

print()
print("Trace for 'madam':")
print("  is_palindrome('madam') → 'm'=='m' → is_palindrome('ada')")
print("  is_palindrome('ada')   → 'a'=='a' → is_palindrome('d')")
print("  is_palindrome('d')     → len=1     → True  ✓")

---
## Summary

| Topic | Key result |
|---|---|
| Impurity metrics (2-class) | Split 1 `(35,5)|(25,35)` has highest reduction across all three measures |
| Impurity metrics (4-class) | Entropy most sensitive; misclassification coarsest |
| Gini proof | $1 - \sum p_i^2$ follows directly from $\sum p_i(1-p_i)$ using $\sum p_i = 1$ |
| Fibonacci recursion | Two base cases (F(0)=0, F(1)=1); reduces n by 1 and 2 per call |
| Palindrome recursion | Base case: len ≤ 1; strips outer characters, reduces length by 2 per call |